# Extração de Q&A

Para este exercício, vamos utilizar o MedPT, que é um dataset feitos com interações reais entre pacientes e médicos. Primeiro, vamos entender o dataset completo.

Para isso, vamos usar a lib `duckdb`, que permite executar filtragens em SQL nos dados do dataset no HuggingFace.

In [ ]:
import duckdb
con = duckdb.connect()

medpt = con.sql("""
     SELECT *
       FROM 'hf://datasets/AKCIT/MedPT/**/*.parquet'
""").df()
original_count = medpt['id'].count # Guardar contagem para comparações posteriores
medpt.count()

O dataset tem as colunas de `question` e `answer`, que serão usadas no finetuning. 
Entretanto, o dataset tem **384k+ perguntas**, o que torna necessária uma filtragem dos dados de acordo com os objetivos do exercício. 

Para este trabalho, escolhemos seguir na temática dos dois primeiros Tech Challenges, e tratar da saúde da mulher, focando em avaliação pré e neo natal. Para isso, vamos simular um hospital materninadade, com finetuning, RAG de Documentos e Tools focadas nisso. 

As colunas `medical_speciality`, `condition` e `question_type` permitem investigar filtragens estruturadas que alcancem um corpus de treinamento viável.

Nosso primeiro passo será extrair todas as linhas que contém pelo menos uma especialidade entre 'Ginecologia' e 'Pediatria'. A coluna de especialidades pode possuir múltiplas especialidades, portanto a seleção incluir todas as outras especialidades que "tocam" nas duas principais.

In [ ]:
maternity_medpt = con.sql("""
     SELECT *
       FROM medpt,
            UNNEST(string_split(medical_specialty, ',')) AS t(esp)
      WHERE (medical_specialty ILIKE '%Pediatra%'
         OR medical_specialty ILIKE '%Ginecologista%')
""").df()
maternity_medpt.count()


Nosso primeiro 
Vamos ver como os exemplos se distribuem entre diferentes especialidas

In [ ]:
speciality_ranking = con.sql("""
     SELECT trim(esp) AS especialidade,
           COUNT(*) AS n
       FROM medpt,
            UNNEST(string_split(medical_specialty, ',')) AS t(esp)
      WHERE (medical_specialty ILIKE '%Pediatra%'
         OR medical_specialty ILIKE '%Ginecologista%')
      GROUP BY 1
      ORDER BY 2 DESC
""").df()
print(speciality_ranking.head())

analogous_specilities = speciality_ranking['especialidade'].tolist()

print(f"\n{len(analogous_specilities)} especialidades diferentes na seleção")

In [ ]:
speciality_ranking["acum"] = (
    speciality_ranking["n"].cumsum() / speciality_ranking["n"].sum()
)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 4))
speciality_ranking["acum"].plot(ax=ax, marker="o", markersize=3)
ax.axhline(0.8, ls="--", lw=1)
ax.axhline(0.9, ls="--", lw=1)
ax.set_xticks(range(len(speciality_ranking)))
ax.set_xticklabels(speciality_ranking["especialidade"], rotation=90, fontsize=7)
ax.set_ylabel("Cobertura Acumulada")
plt.tight_layout()

In [ ]:
conditions_mspecialtys = con.sql("""
    SELECT 
          condition,
          medical_specialty,
          COUNT(*) AS n,
     FROM medpt
     WHERE (medical_specialty ILIKE '%Pediatra%'
     OR medical_specialty ILIKE '%Ginecologista%')
     GROUP BY 1, 2
     ORDER BY n DESC
""").df()
conditions_mspecialtys.describe()


In [ ]:
conditions_mspecialtys.head(10)

In [ ]:
pairs_df = conditions_mspecialtys[['condition', 'medical_specialty']]
pairs = list(zip(pairs_df['condition'], pairs_df['medical_specialty']))


In [ ]:
corpus = con.sql("""
    SELECT m.*
      FROM medpt AS m
      JOIN pairs_df AS p USING (condition, medical_specialty)
""").df()

len(corpus)


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import openai

from openai import AsyncOpenAI
Z_API_KEY = os.getenv("Z_API_KEY")

# Create Z.AI client
client = AsyncOpenAI(
    api_key=Z_API_KEY,
    base_url="https://api.z.ai/api/paas/v4/"
)
print(Z_API_KEY)


In [ ]:
schema = {
    "type": "object",
    "properties": {
        "escopo": {
            "type": "integer",
            "description": "1 se a condicao pertence ao universo da maternidade, 0 caso contrario",
        }
    },
    "required": ["escopo"],
}

In [ ]:
print(pairs[:10])

In [ ]:
import asyncio
import json
import re
from typing import Tuple

sem = asyncio.Semaphore(50)
lock = asyncio.Lock()

def parse_json(texto: str):
    t = texto.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t).strip()
    return json.loads(t)

schema = {
    "type": "object",
    "additionalProperties": {"type": "integer", "enum": [0, 1]},
}

SYSTEM = (
    "Voce e um medico que atua em um hospital maternidade, acompanhando "
    "gestantes, puerperas e bebes ate 1 ano.\n\n"
    "Tarefa: decidir se uma condicao medica pertence ao escopo de atendimento "
    "desse hospital.\n\n"
    "O escopo cobre:\n"
    "- fertilidade e planejamento reprodutivo\n"
    "- gestacao e suas complicacoes\n"
    "- parto\n"
    "- puerperio e amamentacao\n"
    "- bebe ate 1 ano de idade\n\n"
    "CRITERIO. Responda 1 quando pelo menos uma for verdadeira:\n"
    "(a) a condicao so ocorre na gestacao, no parto, no puerperio ou no "
    "recem-nascido;\n"
    "(b) a condicao ocorre tambem fora desse contexto, mas o diagnostico, a "
    "conduta ou o risco mudam de forma relevante quando a paciente esta "
    "gestante ou quando o paciente e um bebe ate 1 ano.\n\n"
    "Responda 0 quando a condicao apenas pode coexistir com a gestacao ou com "
    "o primeiro ano de vida sem que isso altere o manejo, e quando ela nao "
    "pertence a nenhuma das fases acima.\n\n"
    "PAPEL DA ESPECIALIDADE. A condicao decide primeiro. Se a condicao "
    "pertence ao escopo por si mesma, responda 1 independentemente da "
    "especialidade listada. Use a especialidade apenas para desempatar "
    "condicoes ambiguas, que podem ou nao ser do escopo dependendo do "
    "paciente atendido.\n\n"
    "Exemplos:\n"
    'Condicao: Pre-eclampsia | Especialidade: Ginecologista -> {"relevante": 1}\n'
    'Condicao: Asfixia Neonatal | Especialidade: Oncologista, Pediatra -> {"relevante": 1}\n'
    'Condicao: Infeccao Urinaria | Especialidade: Ginecologista -> {"relevante": 1}\n'
    'Condicao: Mioma | Especialidade: Ginecologista -> {"relevante": 1}\n'
    'Condicao: Constipacao em bebes | Especialidade: Pediatra -> {"relevante": 1}\n'
    'Condicao: Menopausa | Especialidade: Ginecologista -> {"relevante": 0}\n'
    'Condicao: Escoliose | Especialidade: Pediatra -> {"relevante": 0}\n'
    'Condicao: Labirintite | Especialidade: Ginecologista -> {"relevante": 0}\n\n'
    "Responda apenas o JSON, sem texto antes ou depois."
)

async def classify(pair: tuple[str, str]) -> bool:
    async with sem:
        cond, esp = pair
        resp = await client.chat.completions.create(
            model="glm-5.3-flash",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM,
                },
                {"role": "user", "content": f"Condicao: {cond} | Especialidade: {esp}"}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "classificacao", "schema": schema},
            },
            temperature=0,
        )
        bruto = resp.choices[0].message.content

    resultado = parse_json(bruto)
    if resultado['relevante']:
        print(f"{cond} - {esp}: Aprovado✅")
        return True
    print(f"{cond} - {esp}: Reprovado⛔")
    return False
    

async def run(pairs: list[Tuple[str, str]]):
    resultados = await asyncio.gather(*(classify(pair) for pair in pairs))
    return [pair for pair, relevant in zip(pairs, resultados) if relevant]



In [ ]:
from pathlib import Path

pair_filename = "pares_selecionados"
p = Path(f"{pair_filename}.json")

selected = None
if p.is_file():
    with p.open('r', encoding="utf-8") as f:
        selected = [tuple(x) for x in json.load(f)]


if not selected:    
    selected = await run(pairs)
    p.write_text(
        json.dumps(selected, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

In [ ]:
import pandas as pd
rejected = list(set(pairs) - set(selected))

selected_count = len(selected)
rejected_count = len(rejected)

print(f"""
{selected_count} pares selecionados
{rejected_count} pares rejeitados
""")



In [ ]:
selected_df = pd.DataFrame(selected, columns=["condition", "medical_specialty"])

corpus = medpt.merge(selected_df, on=["condition", "medical_specialty"], how="inner")

example_count = corpus.count()
condition_count = corpus["condition"].nunique()
specialty_count = corpus["medical_specialty"].nunique()

print(f"""
{example_count} exemplos filtrados em {condition_count} condições e {specialty_count} combinações de especialidades
""")


##Filtragem por tamanho

In [ ]:
corpus["tam_q"] = corpus["question"].str.len()
corpus["tam_a"] = corpus["answer"].str.len()

corpus[["tam_q", "tam_a"]].describe()
corpus.groupby(["condition", "medical_specialty"]).size().sort_values(ascending=False).describe()

In [ ]:
filtrado = corpus[
    corpus["tam_a"].between(100, 3000) & (corpus["tam_q"] >= 20)
]
corpus = corpus.drop_duplicates(subset=["question", "answer"])
print(len(corpus), "→", len(filtrado))

In [ ]:
corpus["question_type"].value_counts()

In [ ]:
pd.crosstab(corpus["condition"], corpus["question_type"]).loc[
    ["HPV", "Gravidez", "Mioma", "Sífilis"]
]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import AgglomerativeClustering

modelo = SentenceTransformer("intfloat/multilingual-e5-small")
g = corpus[(corpus.condition == "Gravidez") &
           (corpus.question_type == "Diagnóstico") &
           (corpus.medical_specialty == "Ginecologista")]

emb = modelo.encode(g["question"].tolist(), normalize_embeddings=True)

cl = AgglomerativeClustering(n_clusters=None, distance_threshold=0.08,
                             metric="cosine", linkage="complete").fit(emb)

g2 = g.assign(cluster=cl.labels_)
print(g2["cluster"].nunique(), "clusters em", len(g))

for c in g2["cluster"].value_counts().head(3).index:
    print(f"\n=== cluster {c} ({(g2.cluster == c).sum()} itens) ===")
    for q in g2[g2.cluster == c]["question"].head(8):
        print("-", q[:110])

In [ ]:
def dedup_grupo(g, limiar=0.08):
    if len(g) < 3:
        return g
    emb = modelo.encode(g["question"].tolist(), normalize_embeddings=True)
    cl = AgglomerativeClustering(
        n_clusters=None, distance_threshold=limiar,
        metric="cosine", linkage="complete",
    ).fit(emb)
    g = g.assign(cluster=cl.labels_)
    return g.loc[g.groupby("cluster")["tam_a"].idxmax()]

corpus_bruto = corpus.copy()
corpus = corpus.drop_duplicates(subset=["question"])  # Tira questões duplicadas

reduzido = (
    corpus.groupby(["condition", "medical_specialty", "question_type"], group_keys=True)
          .apply(dedup_grupo)
          .reset_index()
          .drop(columns=["level_3", "cluster"], errors="ignore")
)

print(len(corpus), "→", len(reduzido))
print(reduzido.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

cont = reduzido["condition"].value_counts()
chave = reduzido["condition"].where(reduzido["condition"].map(cont) >= 15, "raras")

treino, resto = train_test_split(
    reduzido, test_size=0.2, random_state=42, stratify=chave
)
val, teste = train_test_split(
    resto, test_size=0.5, random_state=42, stratify=chave.loc[resto.index]
)

print(len(treino), len(val), len(teste))
print(teste["condition"].nunique(), "condições distintas no teste")

In [ ]:
RARAS_OBST = ["Hemorragia Pós-Parto", "Trabalho De Parto Prematuro",
              "Abortamento Incompleto", "Ameaça De Abortamento", "Gravidez Tubária"]
for c in RARAS_OBST:
    onde = []
    if c in treino["condition"].values: onde.append("treino")
    if c in val["condition"].values: onde.append("val")
    if c in teste["condition"].values: onde.append("teste")
    print(f"{c}: {onde}")

39 e 43 poderiam ser um só. São claramente a mesma pergunta. Com 0.08 eles ficaram separados, o que significa que o limiar está até conservador. Isso é bom: erra pro lado de preservar.

In [ ]:
reduzido = reduzido.reset_index()
reduzido["condition"].value_counts().head(10)

# Reescrita
Precisa ser de médico pra médico, e não de paciente pra médico.


In [ ]:
schema = {
    "type": "object",
    "properties": {
        "acao": {
            "type": "string",
            "enum": ["reescrever", "manter", "descartar"],
        },
        "pergunta": {"type": "string"},
        "motivo": {"type": "string"},
    },
    "required": ["acao", "pergunta", "motivo"],
}

In [ ]:
SYSTEM_REESCRITA = (
    "Voce adapta perguntas medicas para o registro de comunicacao entre "
    "profissionais de saude, em um hospital maternidade.\n\n"
    "As perguntas originais foram escritas por pacientes leigas. Decida uma "
    "das tres acoes:\n\n"
    "reescrever: a pergunta e clinicamente util, mas esta em linguagem leiga "
    "ou em relato pessoal. Reescreva como um profissional perguntaria a um "
    "assistente clinico, sobre conduta, criterio ou manejo. Preserve "
    "exatamente o assunto clinico, sem adicionar nem remover informacao. "
    "Entre 8 e 40 palavras, terminando em ponto de interrogacao.\n\n"
    "manter: a pergunta ja esta em registro tecnico ou impessoal adequado. "
    "Devolva o texto original sem alteracao.\n\n"
    "descartar: a pergunta nao serve. Use quando ela pedir interpretacao de "
    "um valor de exame especifico, quando estiver truncada ou incompreensivel, "
    "ou quando nao houver conteudo clinico. Devolva o texto original.\n\n"
    "Exemplos:\n"
    'Original: to gravida de 8 semanas e tive um sangramento marrom, e normal isso?\n'
    '{"acao": "reescrever", "pergunta": "Qual a conduta diante de sangramento '
    'vaginal escuro em gestante de 8 semanas?", "motivo": "relato pessoal em linguagem leiga"}\n\n'
    'Original: Meu beta hcg deu 21888,0 mUIml. E positivo ou negativo?\n'
    '{"acao": "descartar", "pergunta": "Meu beta hcg deu 21888,0 mUIml. E positivo ou negativo?", '
    '"motivo": "interpretacao de valor isolado de exame"}\n\n'
    'Original: Quais sao os criterios diagnosticos da pre-eclampsia grave?\n'
    '{"acao": "manter", "pergunta": "Quais sao os criterios diagnosticos da pre-eclampsia grave?", '
    '"motivo": "ja em registro tecnico"}\n\n'
    "Responda apenas o JSON, sem texto antes ou depois."
)

In [ ]:
import random

async def reescreve(row, tentativas=4):
    async with sem:
        for i in range(tentativas):
            try:
                resp = await client.chat.completions.create(
                    model="glm-4.7-flashx",
                    messages=[
                        {"role": "system", "content": SYSTEM_REESCRITA},
                        {"role": "user", "content": f"Original: {row['question']}"},
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {"name": "reescrita", "schema": schema},
                    },
                    temperature=0.3,
                )
                r = parse_json(resp.choices[0].message.content)
                return {
                    "id": row["id"],
                    "question_original": row["question"],
                    "question": r["pergunta"],
                    "acao": r["acao"],
                    "motivo": r.get("motivo", ""),
                }
            except Exception as e:
                print(e)
                if i == tentativas - 1:
                    return {"id": row["id"], "erro": type(e).__name__}
                await asyncio.sleep(2 ** i + random.random())


async def run_reescrita(df):
    tasks = [reescreve(r) for _, r in df.iterrows()]
    return await asyncio.gather(*tasks)


amostra = treino.sample(30, random_state=42)
resultados = await run_reescrita(amostra)

In [ ]:
resultados